In [6]:
import folium
import numpy as np
import pandas as pd
import json

m = folium.Map(
    location=[20.67, -103.35],
    zoom_start=12,
    tiles="https://basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}.png?key=cb1_54ff_1_95f1680968a40893034231b1",
    attr='© <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors, '
         '© <a href="https://carto.com/attribution/">CARTO</a>',
)

df = pd.read_csv("./Data/Clean/ALL_Venta.csv")

# CP: drop missing, then 45400.0 -> 45400 so it matches the GeoJSON
df = df.dropna(subset=["CP"])
df["CP"] = df["CP"].astype(int)

# Median price per CP, in millions of pesos
medians = (
    df.groupby("CP")["Price"].median()
      .div(1_000_000)
      .reset_index(name="Median")
)

# Bins in millions; the last edge must be >= the largest median
bins = [0, 1, 1.5, 2, 3, 4, 5, 6, 7, 8, medians["Median"].max() + 0.01]
with open("./Data/Clean/colonies.geojson", encoding="utf-8") as f:
    geo = json.load(f)

median_by_cp = medians.set_index("CP")["Median"].to_dict()

for feature in geo["features"]:
    props = feature["properties"]
    value = median_by_cp.get(props["CP"])
    props["Median"] = f"${value:,.2f} M" if value is not None else "Sin datos"

choropleth = folium.Choropleth(
    geo_data=geo,                     # the modified dict, not the file path
    data=medians,
    columns=["CP", "Median"],
    key_on="feature.properties.CP",
    bins=bins,
    fill_color="RdYlGn_r",
    fill_opacity=0.8,
    line_opacity=0.3,
    nan_fill_color="white",
    legend_name="Precio mediano ZMG (millones MXN)",
).add_to(m)

folium.GeoJsonTooltip(
    fields=["CP", "Colony", "Median"],
    aliases=["CP:", "Colonia:", "Mediana:"],
    localize=True,
    sticky=True,
).add_to(choropleth.geojson)

m.save("test.html")

In [98]:
response = requests.get("http://localhost:8080/colonias/?nombre=Artesanos&municipio_id=39")
content = response.json()
print(content)


{'detalle': 'ajusta tus filtros e intenta de nuevo', 'error': 'no se encontraron resultados'}
